<a href="https://colab.research.google.com/github/kalanakotawalagedara/Grid-box-identify/blob/main/Grid_Box_Prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **1) Identify pocket coordinates of protein based on co-crystalize ligand**


*   Type pdb ID
*   Obtain the center XYZ coordinates based on co-crystalized ligand


*   Not work for apo-proteins





In [ ]:
# @title
import os, math, json, textwrap, re
from collections import defaultdict
from statistics import mean

# -------------------- User parameters --------------------
pdb_id = input("Enter PDB ID (e.g., 3ERT): ") or "3ERT"  # User input for PDB ID

# Validate PDB ID format
if not re.fullmatch(r'^[0-9][A-Za-z0-9]{3}$|^[A-Za-z0-9]{4}$', pdb_id):
    print(f"Error: Invalid PDB ID format '{pdb_id}'. PDB IDs typically consist of 4 alphanumeric characters (e.g., '3ERT').")
    download_success = False
    pdb_text = None
    master = {"pdb_id": pdb_id, "downloaded": download_success, "ligands": []}
else:
    output_dir = "/mnt/data"
    recommended_default_box = (30.0, 30.0, 30.0)  # default focused box (\u00C5)
    vina_exhaustiveness = 50
    contact_cutoff = 4.5  # \u00C5 for protein-ligand contacts
    # Common ignore list for glycans, solvents, and crystallization agents
    COMMON_IGNORE = {'HOH','WAT','NA','CL','K','MG','CA','SO4','PO4','EDO','GOL','DMS','MPD','ACE','NAG','SO3','ZN','MN','FE','NI','IOD','BMA', 'NDG', 'MAN', 'BOG'}

    # create output dir
    os.makedirs(output_dir, exist_ok=True)

    # -------------------- Parsing helpers --------------------
    def parse_pdb_text(pdb_text):
        protein_atoms = []  # list of dicts {'chain','resseq','resname','atom','x','y','z'}
        het_atoms_by_res = defaultdict(list)  # key: (resname, chain, resseq) -> list of atom dicts
        for line in pdb_text.splitlines():
            if len(line) < 54:
                continue
            record = line[0:6].strip()
            if record in ("ATOM","HETATM"):
                atom_name = line[12:16].strip()
                resname = line[17:20].strip()
                chain = line[21].strip() or "_"
                resseq = line[22:26].strip()
                try:
                    x = float(line[30:38].strip())
                    y = float(line[38:46].strip())
                    z = float(line[46:54].strip())
                except Exception:
                    continue
                entry = {"atom": atom_name, "resname": resname, "chain": chain, "resseq": resseq, "x": x, "y": y, "z": z}
                if record == "ATOM":
                    protein_atoms.append(entry)
                else:
                    het_atoms_by_res[(resname, chain, resseq)].append(entry)
        return protein_atoms, het_atoms_by_res

    def compute_centroid(atom_list):
        xs = [a['x'] for a in atom_list]
        ys = [a['y'] for a in atom_list]
        zs = [a['z'] for a in atom_list]
        return (mean(xs), mean(ys), mean(zs))

    def ligand_extent(atom_list):
        xs = [a['x'] for a in atom_list]
        ys = [a['y'] for a in atom_list]
        zs = [a['z'] for a in atom_list]
        extent_x = max(xs)-min(xs) if xs else 0.0
        extent_y = max(ys)-min(ys) if ys else 0.0
        extent_z = max(zs)-min(zs) if zs else 0.0
        return extent_x, extent_y, extent_z

    def distance(a,b):
        return math.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2 + (a[2]-b[2])**2)

    def ligand_protein_contacts(lig_atoms, prot_atoms, cutoff=4.5):
        contacts = set()
        contact_list = []
        for la in lig_atoms:
            lcoord = (la['x'], la['y'], la['z'])
            for pa in prot_atoms:
                pcoord = (pa['x'], pa['y'], pa['z'])
                if distance(lcoord, pcoord) <= cutoff:
                    contacts.add((pa['resname'], pa['chain'], pa['resseq']))
                    contact_list.append(pa)
        return contacts, contact_list

hydrophobic_residues = {"ALA","VAL","ILE","LEU","PHE","TRP","TYR","MET"}

def classify_site(contacts, contact_atoms):
    num_contacts = len(contacts)
    if contact_atoms:
        resnames = [a['resname'] for a in contact_atoms]
        hydrophobic_count = sum(1 for r in resnames if r.upper() in hydrophobic_residues)
        hydrophobic_fraction = hydrophobic_count / len(resnames)
    else:
        hydrophobic_fraction = 0.0
    if num_contacts >= 8 and hydrophobic_fraction >= 0.4:
        classification = "Likely orthosteric (buried in hydrophobic pocket)"
    elif num_contacts >= 8:
        classification = "Likely orthosteric (buried)"
    elif 3 <= num_contacts <= 7:
        classification = "Ambiguous (possible allosteric or shallow orthosteric)"
    elif num_contacts <= 2 and num_contacts > 0:
        classification = "Likely allosteric or peripheral (few contacts)"
    else:
        classification = "No protein contacts detected"
    return classification, num_contacts, hydrophobic_fraction

# -------------------- Fetch PDB --------------------
pdb_text = None
download_success = False
try:
    import requests
    url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    pdb_text = r.text
    download_success = True
except Exception as e:
    print(f"Warning: could not download PDB {pdb_id}.")
    pdb_text = None

# -------------------- Main processing --------------------
master = {"pdb_id": pdb_id, "downloaded": download_success, "ligands": []}

if pdb_text is not None:
    protein_atoms, het_atoms_by_res = parse_pdb_text(pdb_text)
    # Filter to only relevant ligands by checking against the ignore list
    ligand_groups = {k:v for k,v in het_atoms_by_res.items() if k[0].upper() not in COMMON_IGNORE}

    if not ligand_groups:
        print("No relevant ligands found in this PDB file.")

    for (resname, chain, resseq), atoms in ligand_groups.items():
        centroid = compute_centroid(atoms)
        ext_x, ext_y, ext_z = ligand_extent(atoms)
        padding = 8.0
        suggested_size_x = max(recommended_default_box[0], ext_x + padding*2)
        suggested_size_y = max(recommended_default_box[1], ext_y + padding*2)
        suggested_size_z = max(recommended_default_box[2], ext_z + padding*2)
        suggested_size = (round(suggested_size_x,3), round(suggested_size_y,3), round(suggested_size_z,3))
        contacts, contact_atoms = ligand_protein_contacts(atoms, protein_atoms, cutoff=contact_cutoff)
        classification, num_contacts, hydrophobic_fraction = classify_site(contacts, contact_atoms)
        rec = {
            "resname": resname, "chain": chain, "resseq": resseq,
            "centroid": {"x": round(centroid[0],3), "y": round(centroid[1],3), "z": round(centroid[2],3)},
            "ligand_atom_count": len(atoms),
            "extent": {"x": round(ext_x,3), "y": round(ext_y,3), "z": round(ext_z,3)},
            "recommended_box": {"size_x": suggested_size[0], "size_y": suggested_size[1], "size_z": suggested_size[2]},
            "classification": classification,
            "num_contacts": num_contacts,
            "hydrophobic_fraction": round(hydrophobic_fraction,3)
        }
        master["ligands"].append(rec)

# -------------------- Write outputs --------------------
master_txt_lines = [f"# Master docking grid report for PDB: {pdb_id}", f"# Downloaded: {download_success}", ""]
for lig in master["ligands"]:
    master_txt_lines.append(f"--- Ligand: {lig['resname']} (chain {lig['chain']}, resseq {lig['resseq']}) ---")
    master_txt_lines.append(f"Binding site center (\u00C5): {lig['centroid']['x']}, {lig['centroid']['y']}, {lig['centroid']['z']}")
    master_txt_lines.append(f"Search space size (\u00C5): {lig['recommended_box']['size_x']}, {lig['recommended_box']['size_y']}, {lig['recommended_box']['size_z']}")
    master_txt_lines.append(f"Contacts: {lig['num_contacts']}")
    master_txt_lines.append(f"Site: {lig['classification']}")
    master_txt_lines.append("")

master_txt = "\n".join(master_txt_lines)
master_txt_path = os.path.join(output_dir, f"master_report_{pdb_id}.txt")
master_json_path = os.path.join(output_dir, f"master_report_{pdb_id}.json")
with open(master_txt_path, "w") as f: f.write(master_txt)
with open(master_json_path, "w") as f: json.dump(master, f, indent=2)
created_files = [master_txt_path, master_json_path]
for lig in master["ligands"]:
    safe_label = f"{pdb_id}_{lig['resname']}_{lig['chain']}_{lig['resseq']}"
    vina_conf = textwrap.dedent(f"""
    center_x = {lig['centroid']['x']}
    center_y = {lig['centroid']['y']}
    center_z = {lig['centroid']['z']}
    size_x = {lig['recommended_box']['size_x']}
    size_y = {lig['recommended_box']['size_y']}
    size_z = {lig['recommended_box']['size_z']}
    exhaustiveness = {vina_exhaustiveness}
    """).strip()
    vina_conf_path = os.path.join(output_dir, f"vina_conf_{safe_label}.conf")
    with open(vina_conf_path, "w") as f: f.write(vina_conf)
    created_files.append(vina_conf_path)

In [ ]:
# @title
import pandas as pd

# Prepare data for CSV
csv_data = []
for lig in master['ligands']:
    csv_data.append({
        'pdb_id': master['pdb_id'],
        'resname': lig['resname'],
        'chain': lig['chain'],
        'resseq': lig['resseq'],
        'center_x': lig['centroid']['x'],
        'center_y': lig['centroid']['y'],
        'center_z': lig['centroid']['z'],
        'size_x': lig['recommended_box']['size_x'],
        'size_y': lig['recommended_box']['size_y'],
        'size_z': lig['recommended_box']['size_z']
    })

# Create DataFrame
ligand_coords_df = pd.DataFrame(csv_data)

# Define the output path for the CSV file
csv_output_path = os.path.join(output_dir, f"ligand_grid_coordinates_{pdb_id}.csv")

# Save to CSV
ligand_coords_df.to_csv(csv_output_path, index=False)

print(f"CSV file created: {csv_output_path}")

# Display the DataFrame head
display(ligand_coords_df.head())

# Add to created files list for download links
created_files.append(csv_output_path)

# **2) Identify pocket coordinates from apo-protein**


*   Install Fpocket
*   Upload .pdb file (better to use Apo-protein)



In [ ]:
# @title Install P2Rank
import os

# 1. Update package lists and install Java (P2Rank requirement)
print("Updating package lists and installing Java...")
!apt-get update -qq
!apt-get install -y openjdk-11-jdk-headless -qq > /dev/null

# 2. Download and unzip P2Rank
if not os.path.exists('p2rank_2.4.1'):
    print("Downloading P2Rank...")
    !wget -q https://github.com/rdk/p2rank/releases/download/2.4.1/p2rank_2.4.1.tar.gz
    !tar -xzf p2rank_2.4.1.tar.gz
    !rm p2rank_2.4.1.tar.gz

print("P2Rank 2.4.1 installation environment prepared.")

In [ ]:
# @title P2Rank — Pocket Prediction + Binding Residue Extraction
import os, re, shutil, subprocess, sys
import numpy as np
import pandas as pd
from google.colab import files

print("=== P2Rank: Pocket + Binding Residue Analysis ===")

# ── 1. Install P2Rank if missing ─────────────────────────────────────
PRANK_VERSION = "2.4.1"
PRANK_DIR     = f"p2rank_{PRANK_VERSION}"
PRANK_SCRIPT  = os.path.join(os.getcwd(), PRANK_DIR, "prank")

if not os.path.isfile(PRANK_SCRIPT):
    print(f"Downloading P2Rank v{PRANK_VERSION}...")
    dl = subprocess.run(
        f"wget -q https://github.com/rdk/p2rank/releases/download/{PRANK_VERSION}/"
        f"p2rank_{PRANK_VERSION}.tar.gz -O p2rank.tar.gz && "
        f"tar -xzf p2rank.tar.gz && rm p2rank.tar.gz",
        shell=True, capture_output=True, text=True, executable="/bin/bash"
    )
    if dl.returncode != 0 or not os.path.isfile(PRANK_SCRIPT):
        print("Download FAILED:\n", dl.stderr); sys.exit(1)
    print("P2Rank ready.")
os.chmod(PRANK_SCRIPT, 0o755)

# ── 2. Upload PDB ─────────────────────────────────────────────────────
print("\nUpload your PDB file (e.g. 3N8W.pdb):")
uploaded = files.upload()
if not uploaded:
    sys.exit("No file uploaded.")

original_name = list(uploaded.keys())[0]
safe_base = re.sub(r'[^a-zA-Z0-9_\-]', '_', original_name.rsplit('.', 1)[0]).strip('_')
safe_name = f"{safe_base}.pdb"

if original_name != safe_name:
    shutil.copy(original_name, safe_name)
    print(f"Renamed: '{original_name}' → '{safe_name}'")

input_path = os.path.join(os.getcwd(), safe_name)
output_dir = os.path.join(os.getcwd(), "p2rank_output")
os.makedirs(output_dir, exist_ok=True)

# ── 3. Run P2Rank ─────────────────────────────────────────────────────
print(f"\nRunning P2Rank on '{safe_name}'...")
result = subprocess.run(
    f'"{PRANK_SCRIPT}" predict -f "{input_path}" -o "{output_dir}"',
    shell=True, capture_output=True, text=True, executable="/bin/bash"
)
if result.returncode != 0:
    print("P2RANK FAILED\nstdout:", result.stdout, "\nstderr:", result.stderr)
    sys.exit(1)
print("P2Rank completed.")

# ── 4. Find output files ──────────────────────────────────────────────
def find_file(root, suffix):
    for dirpath, _, fnames in os.walk(root):
        for f in fnames:
            if f.endswith(suffix):
                return os.path.join(dirpath, f)
    return None

pred_csv = find_file(output_dir, "_predictions.csv")
res_csv  = find_file(output_dir, "_residues.csv")

if not pred_csv:
    print("ERROR: _predictions.csv not found.")
    print("All output files:"); [print(" ", os.path.join(r,f))
        for r,_,fs in os.walk(output_dir) for f in fs]
    sys.exit(1)

print(f"\nPredictions : {pred_csv}")
print(f"Residues    : {res_csv}")

# ── 5. Parse predictions CSV ──────────────────────────────────────────
pred_df = pd.read_csv(pred_csv)
pred_df.columns = [c.strip() for c in pred_df.columns]
pred_df = pred_df.apply(lambda c: c.str.strip() if c.dtype == object else c)

for col in ['score', 'probability', 'center_x', 'center_y', 'center_z']:
    if col in pred_df.columns:
        pred_df[col] = pd.to_numeric(pred_df[col], errors='coerce')

print("\n--- All Predicted Pockets ---")
display_cols = [c for c in ['name','rank','score','probability',
                             'center_x','center_y','center_z'] if c in pred_df.columns]
display(pred_df[display_cols])

# ── 6. Parse residues CSV ─────────────────────────────────────────────
res_df = None
if res_csv:
    res_df = pd.read_csv(res_csv)
    res_df.columns = [c.strip() for c in res_df.columns]
    res_df = res_df.apply(lambda c: c.str.strip() if c.dtype == object else c)
    for col in ['score', 'probability']:
        if col in res_df.columns:
            res_df[col] = pd.to_numeric(res_df[col], errors='coerce')

# ── 7. Ligand coordinate lookup (optional — enter known XYZ) ─────────
# If you know your ligand's approximate center (e.g. from PyMOL or PDB),
# enter it below to auto-identify the closest predicted pocket.
# For 3N8W FLP, coordinates are approximately: X=10.5, Y=65.2, Z=15.8
# Leave as None to skip auto-matching and inspect all pockets manually.

LIGAND_NAME = "FLP"          # just a label — for display
LIGAND_X    = None           # e.g. 10.5  ← replace with actual value
LIGAND_Y    = None           # e.g. 65.2
LIGAND_Z    = None           # e.g. 15.8

matched_pocket_name = None

if all(v is not None for v in [LIGAND_X, LIGAND_Y, LIGAND_Z]):
    if {'center_x','center_y','center_z'}.issubset(pred_df.columns):
        dists = np.sqrt(
            (pred_df['center_x'] - LIGAND_X)**2 +
            (pred_df['center_y'] - LIGAND_Y)**2 +
            (pred_df['center_z'] - LIGAND_Z)**2
        )
        idx    = dists.idxmin()
        dist_A = dists[idx]
        matched_pocket_name = pred_df.loc[idx, 'name'] if 'name' in pred_df.columns else str(idx+1)

        print(f"\n--- Closest Pocket to {LIGAND_NAME} ---")
        print(f"Pocket : {matched_pocket_name}")
        print(f"Distance to ligand center : {dist_A:.2f} Å")
        display(pred_df.loc[[idx], display_cols])

# ── 8. Extract binding residues per pocket ────────────────────────────
print("\n--- Binding Residues Per Pocket ---")

if res_df is not None and 'pocket' in res_df.columns:
    # _residues.csv has a 'pocket' column mapping residues → pocket number
    # Show top pockets (up to 5) sorted by score
    top_n = min(5, len(pred_df))
    top_pockets = pred_df.head(top_n)

    for _, row in top_pockets.iterrows():
        pocket_name = row.get('name', '?')
        # pocket column in residues CSV is usually "pocket1", "pocket2" etc.
        pocket_num  = re.search(r'\d+', str(pocket_name))
        pocket_num  = pocket_num.group() if pocket_num else None

        if pocket_num is None:
            continue

        mask = res_df['pocket'].astype(str).str.contains(pocket_num, na=False)
        pocket_residues = res_df[mask].copy()

        if pocket_residues.empty:
            continue

        prob_col  = 'probability' if 'probability' in pocket_residues.columns else None
        score_col = 'score'       if 'score'       in pocket_residues.columns else None
        name_col  = next((c for c in ['residue_name','name','res_name','residue']
                          if c in pocket_residues.columns), None)
        chain_col = next((c for c in ['chain','chain_id'] if c in pocket_residues.columns), None)
        seqnum_col= next((c for c in ['residue_sequence_number','seq_num','res_num','seqnum']
                          if c in pocket_residues.columns), None)

        show_cols = [c for c in [name_col, chain_col, seqnum_col, score_col, prob_col]
                     if c is not None]

        header = f"Pocket {pocket_name}"
        if 'score' in row and pd.notna(row['score']):
            header += f"  |  score={row['score']:.3f}"
        if 'probability' in row and pd.notna(row['probability']):
            header += f"  |  probability={row['probability']:.3f}"
        if matched_pocket_name and pocket_name == matched_pocket_name:
            header += f"  ← closest to {LIGAND_NAME}"

        print(f"\n{'─'*60}")
        print(header)
        print(f"{'─'*60}")
        if show_cols:
            display(pocket_residues[show_cols].sort_values(
                score_col, ascending=False) if score_col else pocket_residues[show_cols])
        else:
            display(pocket_residues)

elif 'residue_ids' in pred_df.columns:
    # Fallback: residue IDs are embedded inside predictions CSV as a string list
    print("(Using residue_ids column from predictions CSV — residues.csv not available)")
    for _, row in pred_df.head(5).iterrows():
        print(f"\nPocket {row.get('name','?')} : {row['residue_ids']}")

else:
    print("Residue data not available. Ensure P2Rank v2.3+ is installed.")

# ── 9. Save all outputs ───────────────────────────────────────────────
out_pred = f"p2rank_pockets_{safe_base}.csv"
pred_df.to_csv(out_pred, index=False)
files.download(out_pred)

if res_df is not None:
    out_res = f"p2rank_residues_{safe_base}.csv"
    res_df.to_csv(out_res, index=False)
    files.download(out_res)
    print(f"\nDownloaded: {out_pred}, {out_res}")
else:
    print(f"\nDownloaded: {out_pred}")

In [ ]:
# @title Template-Based Binding Site Annotation
# ── What this does ───────────────────────────────────────────────────
# 1. Auto-detects the PDB ID and all ligands from the uploaded PDB file
# 2. Fetches experimental contact residues directly from the PDB file
#    (does NOT rely on SITE records, which are often missing)
# 3. Fetches 3DLigandSite + P2Rank residues from PDBe-KB Graph API
#    using correct case-insensitive accession matching
# 4. Cross-references every P2Rank pocket against all ligands by
#    computing nearest-pocket distance — zero hardcoding
# 5. Outputs one residue table per ligand, all three sources side-by-side
# ─────────────────────────────────────────────────────────────────────

import os, re, glob, time, json
import numpy as np
import pandas as pd
import requests
from google.colab import files

print("=== Template-Based Binding Site Annotation ===")
print("Sources: PDB contacts · 3DLigandSite · P2Rank (PDBe-KB)\n")

CONTACT_CUTOFF_A = 4.5   # Å — atom distance for "binding residue" from PDB file
MATCH_RADIUS_A   = 8.0   # Å — pocket centre → ligand centre for pocket matching

# ══════════════════════════════════════════════════════════════════════
# BLOCK 0 — Load P2Rank results from previous cell (or from CSV files)
# ══════════════════════════════════════════════════════════════════════
try:
    _ = pred_df
    print("pred_df loaded from P2Rank cell.")
except NameError:
    csvs = glob.glob("p2rank_pockets_*.csv")
    pred_df = pd.read_csv(csvs[0]) if csvs else pd.DataFrame()
    if not pred_df.empty:
        pred_df.columns = [c.strip() for c in pred_df.columns]
        print(f"pred_df loaded from {csvs[0]}")

try:
    _ = res_df
    print("res_df loaded from P2Rank cell.")
except NameError:
    csvs = glob.glob("p2rank_residues_*.csv")
    res_df = pd.read_csv(csvs[0]) if csvs else None
    if res_df is not None:
        res_df.columns = [c.strip() for c in res_df.columns]
        print(f"res_df loaded from {csvs[0]}")

# Coerce numeric columns in pred_df
if not pred_df.empty:
    for col in ["score","probability","center_x","center_y","center_z"]:
        if col in pred_df.columns:
            pred_df[col] = pd.to_numeric(pred_df[col], errors="coerce")

# ══════════════════════════════════════════════════════════════════════
# BLOCK 1 — Find uploaded PDB file and auto-detect PDB ID + all ligands
# ══════════════════════════════════════════════════════════════════════
print("\n" + "─"*60)
print("1. PDB File — Auto-detecting PDB ID and Ligands")
print("─"*60)

# Find the PDB file used in the previous cell
pdb_candidates = glob.glob("*.pdb")
if not pdb_candidates:
    print("No .pdb file found in /content/. Upload one via the P2Rank cell first.")
    raise SystemExit()

pdb_file = pdb_candidates[0]
print(f"Using PDB file: {pdb_file}")

# Auto-detect 4-letter PDB ID from HEADER record or filename
pdb_id = None
with open(pdb_file) as f:
    for line in f:
        if line.startswith("HEADER"):
            # HEADER line: cols 62-65 usually contain the PDB ID
            if len(line) >= 66:
                candidate = line[62:66].strip()
                if re.match(r'^[0-9][A-Za-z0-9]{3}$', candidate):
                    pdb_id = candidate.lower()
                    break
        if line.startswith("REMARK") and "pdb id" in line.lower():
            m = re.search(r'\b([0-9][A-Za-z0-9]{3})\b', line, re.IGNORECASE)
            if m:
                pdb_id = m.group(1).lower()
                break

if not pdb_id:
    # Fall back: use the filename stem if it looks like a PDB ID
    stem = os.path.splitext(os.path.basename(pdb_file))[0]
    if re.match(r'^[0-9][A-Za-z0-9]{3}$', stem):
        pdb_id = stem.lower()
    else:
        pdb_id = stem[:4].lower()

print(f"PDB ID detected: {pdb_id.upper()}")

# ── Parse ALL HETATM ligands directly from PDB file ──────────────────
# HETATM format: cols 17-20 resname, 21 chain, 22-25 resseq, 30-54 xyz
# Exclude water (HOH/WAT) and common crystallographic additives
EXCLUDE = {"HOH","WAT","EDO","PEG","GOL","SO4","PO4","MPD",
           "TRS","MES","ACT","FMT","CIT","EPE","BME","DMS",
           "IOD","CL","NA","MG","ZN","CA","FE","MN","CO"}

ligands_in_pdb = {}   # key: (resname, chain, resnum) → list of xyz arrays
protein_atoms  = []   # list of (resname, chain, resnum, atom_name, xyz)

with open(pdb_file) as f:
    for line in f:
        rec = line[:6].strip()
        if rec == "HETATM":
            resname = line[17:20].strip()
            chain   = line[21].strip()
            try:
                resnum  = int(line[22:26].strip())
                x, y, z = float(line[30:38]), float(line[38:46]), float(line[46:54])
            except ValueError:
                continue
            if resname in EXCLUDE:
                continue
            key = (resname, chain, resnum)
            ligands_in_pdb.setdefault(key, []).append(np.array([x, y, z]))

        elif rec == "ATOM":
            resname = line[17:20].strip()
            chain   = line[21].strip()
            try:
                resnum  = int(line[22:26].strip())
                aname   = line[12:16].strip()
                x, y, z = float(line[30:38]), float(line[38:46]), float(line[46:54])
            except ValueError:
                continue
            protein_atoms.append((resname, chain, resnum, aname,
                                   np.array([x, y, z])))

if not ligands_in_pdb:
    print("No HETATM ligands found in PDB file (excluding water/ions).")
else:
    print(f"\nLigands found in PDB file:")
    lig_rows = []
    for (rn, ch, rnum), atoms in ligands_in_pdb.items():
        ctr = np.mean(atoms, axis=0)
        lig_rows.append({"Ligand": rn, "Chain": ch,
                          "Res num": rnum, "Atom count": len(atoms),
                          "Center X": round(ctr[0],2),
                          "Center Y": round(ctr[1],2),
                          "Center Z": round(ctr[2],2)})
    lig_summary_df = pd.DataFrame(lig_rows)
    display(lig_summary_df)

# ── Compute contact residues per ligand from atomic distances ─────────
print(f"\nComputing contact residues (cutoff = {CONTACT_CUTOFF_A} Å)...")

contact_results = {}   # (resname, chain, resnum) → list of contact residue strings

for lig_key, lig_atoms in ligands_in_pdb.items():
    lig_arr    = np.stack(lig_atoms)          # shape (N_atoms, 3)
    contacts   = {}                            # (prot_resname, chain, resnum) → min_dist

    for p_resname, p_chain, p_resnum, p_aname, p_xyz in protein_atoms:
        dists    = np.linalg.norm(lig_arr - p_xyz, axis=1)
        min_dist = dists.min()
        if min_dist <= CONTACT_CUTOFF_A:
            pk = (p_resname, p_chain, p_resnum)
            if pk not in contacts or contacts[pk] > min_dist:
                contacts[pk] = min_dist

    # Sort by residue number
    sorted_contacts = sorted(contacts.items(), key=lambda x: x[0][2])
    contact_results[lig_key] = [
        f"{r[0]}{r[2]}.{r[1]}" for r, _ in sorted_contacts
    ]

# ══════════════════════════════════════════════════════════════════════
# BLOCK 2 — PDBe-KB Graph API: 3DLigandSite + P2Rank residues
# ══════════════════════════════════════════════════════════════════════
print("\n" + "─"*60)
print("2. PDBe-KB Annotations (3DLigandSite + P2Rank)")
print("─"*60)

BASE_PDBE  = "https://www.ebi.ac.uk/pdbe/api"
BASE_GRAPH = "https://www.ebi.ac.uk/pdbe/graph-api"

def api_get(url, label=""):
    for attempt in range(3):
        try:
            r = requests.get(url, timeout=30)
            if r.status_code == 200:
                return r.json()
            if r.status_code == 404:
                print(f"  [404] {label or url}")
                return None
            print(f"  [HTTP {r.status_code}] {label}")
        except requests.exceptions.RequestException as e:
            print(f"  [attempt {attempt+1} error] {label}: {e}")
            time.sleep(2)
    return None

# Resolve UniProt accession
uniprot_acc = None
uniprot_data = api_get(f"{BASE_PDBE}/mappings/uniprot/{pdb_id}", "UniProt mapping")
if uniprot_data and pdb_id in uniprot_data:
    mappings = uniprot_data[pdb_id].get("UniProt", {})
    if mappings:
        uniprot_acc = next(iter(mappings))
        print(f"UniProt accession: {uniprot_acc}")

# Fetch PDBe-KB annotations — case-insensitive accession matching
tdls_residues   = []
p2rank_residues = []

if uniprot_acc:
    ann_data = api_get(
        f"{BASE_GRAPH}/uniprot/annotations/{uniprot_acc}",
        "PDBe-KB annotations"
    )
    if ann_data and uniprot_acc in ann_data:
        for provider in ann_data[uniprot_acc].get("data", []):
            acc_lower = provider.get("accession", "").lower()
            label     = provider.get("label", provider.get("accession",""))
            residues  = provider.get("residues", [])

            if acc_lower == "3dligandsite":
                tdls_residues = [int(x["startIndex"]) for x in residues
                                 if x.get("startIndex") is not None]
                print(f"3DLigandSite: {len(tdls_residues)} predicted residues")

            elif acc_lower == "p2rank":
                p2rank_residues = [int(x["startIndex"]) for x in residues
                                   if x.get("startIndex") is not None]
                print(f"P2Rank (PDBe-KB): {len(p2rank_residues)} predicted residues")

    if not tdls_residues and not p2rank_residues:
        print("  No 3DLigandSite or P2Rank annotations found for this UniProt accession.")
        print("  Available providers:")
        if ann_data and uniprot_acc in ann_data:
            for p in ann_data[uniprot_acc].get("data", []):
                print(f"    accession='{p.get('accession','')}' "
                      f"label='{p.get('label','')}'  "
                      f"residues={len(p.get('residues',[]))}")
else:
    print("  Could not resolve UniProt accession — skipping PDBe-KB step.")

# ── Map UniProt residue numbers → PDB residue numbers via SIFTS ──────
# PDBe-KB annotations use UniProt numbering; we need PDB numbering to
# compare against the contact residues computed above.
sifts_uniprot_to_pdb = {}   # uniprot_resnum → (pdb_resname, chain, pdb_resnum)

sifts_data = api_get(
    f"{BASE_PDBE}/mappings/all_isoforms/{pdb_id}",
    "SIFTS residue mapping"
)
if sifts_data and pdb_id in sifts_data:
    for uni_acc, uni_info in sifts_data[pdb_id].get("UniProt", {}).items():
        for mapping in uni_info.get("mappings", []):
            uni_start = mapping.get("unp_start")
            uni_end   = mapping.get("unp_end")
            pdb_start = mapping.get("start", {}).get("author_residue_number")
            pdb_end   = mapping.get("end",   {}).get("author_residue_number")
            chain     = mapping.get("chain_id", "A")
            if all(v is not None for v in [uni_start, uni_end, pdb_start, pdb_end]):
                for offset in range(uni_end - uni_start + 1):
                    sifts_uniprot_to_pdb[uni_start + offset] = (
                        chain, pdb_start + offset
                    )

print(f"\nSIFTS mapped {len(sifts_uniprot_to_pdb)} UniProt → PDB residue positions")

def uniprot_to_pdb_residues(uni_resnums, sifts_map):
    """Convert UniProt residue numbers to 'RESNUMchain' strings."""
    result = []
    for u in uni_resnums:
        if u in sifts_map:
            chain, pdb_num = sifts_map[u]
            result.append((chain, pdb_num))
    return sorted(set(result), key=lambda x: x[1])

tdls_pdb   = uniprot_to_pdb_residues(tdls_residues,   sifts_uniprot_to_pdb)
p2rank_pdb = uniprot_to_pdb_residues(p2rank_residues, sifts_uniprot_to_pdb)

print(f"3DLigandSite → {len(tdls_pdb)} PDB-numbered residues after SIFTS mapping")
print(f"P2Rank (PDBe-KB) → {len(p2rank_pdb)} PDB-numbered residues after SIFTS mapping")

# ══════════════════════════════════════════════════════════════════════
# BLOCK 3 — Match P2Rank pockets to ligands by distance
# ══════════════════════════════════════════════════════════════════════
print("\n" + "─"*60)
print("3. P2Rank Pocket → Ligand Matching")
print("─"*60)

# Compute ligand centres
lig_centres = {
    lig_key: np.mean(np.stack(atoms), axis=0)
    for lig_key, atoms in ligands_in_pdb.items()
}

# For each P2Rank pocket, find the nearest ligand
pocket_to_ligand = {}   # pocket_name → (lig_key, dist_A)
if not pred_df.empty and {"center_x","center_y","center_z"}.issubset(pred_df.columns):
    for _, row in pred_df.iterrows():
        pname   = row.get("name", "?")
        p_xyz   = np.array([row["center_x"], row["center_y"], row["center_z"]])
        best_key, best_dist = None, float("inf")
        for lig_key, ctr in lig_centres.items():
            d = np.linalg.norm(p_xyz - ctr)
            if d < best_dist:
                best_dist = d
                best_key  = lig_key
        pocket_to_ligand[pname] = (best_key, round(best_dist, 2))

    # Print matching table
    match_rows = []
    for _, row in pred_df.iterrows():
        pname = row.get("name","?")
        lk, dist = pocket_to_ligand.get(pname, (None, None))
        match_rows.append({
            "Pocket"      : pname,
            "Score"       : round(row.get("score", float("nan")), 3),
            "Probability" : round(row.get("probability", float("nan")), 3),
            "Nearest ligand": f"{lk[0]}.{lk[1]}" if lk else "—",
            "Distance (Å)": dist,
            "Within 8 Å"  : "✓" if dist is not None and dist <= MATCH_RADIUS_A else "✗",
        })
    match_df = pd.DataFrame(match_rows)
    display(match_df)
else:
    print("pred_df missing coordinate columns — skipping pocket matching.")
    match_df = pd.DataFrame()

# ══════════════════════════════════════════════════════════════════════
# BLOCK 4 — Per-ligand consolidated residue table
# ══════════════════════════════════════════════════════════════════════
print("\n" + "─"*60)
print("4. Per-Ligand Binding Residues — All Three Sources")
print("─"*60)

# Build a set of (chain, resnum) for 3DLigandSite and P2Rank-PDBe for comparison
tdls_set   = set(tdls_pdb)
p2rank_set = set(p2rank_pdb)

all_output_dfs = {}

for lig_key, lig_atoms in ligands_in_pdb.items():
    rn, ch, rnum = lig_key
    lig_label = f"{rn} (chain {ch}, res {rnum})"
    print(f"\n{'═'*60}")
    print(f"Ligand: {lig_label}")
    print(f"{'═'*60}")

    lig_ctr = lig_centres[lig_key]

    # Contact residues from PDB (atomic distance)
    pdb_contacts = contact_results.get(lig_key, [])
    # Convert to (chain, resnum) set for overlap calculation
    pdb_set = set()
    for cr in pdb_contacts:
        m = re.match(r'[A-Z]+(\d+)\.([A-Z])', cr)
        if m:
            pdb_set.add((m.group(2), int(m.group(1))))

    # Which P2Rank pocket is closest to this ligand?
    best_pocket_name = None
    best_pocket_dist = float("inf")
    for pname, (lk, dist) in pocket_to_ligand.items():
        if lk == lig_key and dist < best_pocket_dist:
            best_pocket_dist = dist
            best_pocket_name = pname

    # Get P2Rank local residues for the matched pocket
    p2rank_local_residues = []
    if best_pocket_name and res_df is not None:
        # Find the pocket number
        pnum = re.search(r'\d+', best_pocket_name)
        if pnum and "pocket" in res_df.columns:
            mask = res_df["pocket"].astype(str).str.contains(
                pnum.group(), na=False)
            pocket_res = res_df[mask]

            # Detect column names dynamically
            seqnum_col = next((c for c in pocket_res.columns
                if re.search(r'seq|num|resnum|residue_seq|author_res', c, re.I)), None)
            name_col   = next((c for c in pocket_res.columns
                if re.search(r'res.*name|name|aa$', c, re.I)
                and c != seqnum_col), None)
            chain_col  = next((c for c in pocket_res.columns
                if re.search(r'chain', c, re.I)), None)

            print(f"  Matched P2Rank pocket: {best_pocket_name} "
                  f"(distance to ligand = {best_pocket_dist} Å)")
            print(f"  P2Rank residues CSV columns: {pocket_res.columns.tolist()}")

            for _, row in pocket_res.iterrows():
                seqnum = int(pd.to_numeric(row[seqnum_col], errors="coerce")) \
                         if seqnum_col else None
                chain  = row[chain_col].strip() if chain_col else "?"
                name   = row[name_col].strip()  if name_col  else "?"
                if seqnum is not None:
                    p2rank_local_residues.append((name, chain, seqnum))

    # Build unified residue table
    # Collect all unique (chain, resnum) positions across all three sources
    all_positions = set()
    all_positions.update(pdb_set)
    all_positions.update(tdls_set)
    all_positions.update(p2rank_set)
    all_positions.update({(r[1], r[2]) for r in p2rank_local_residues})

    rows = []
    for chain_r, resnum_r in sorted(all_positions, key=lambda x: x[1]):
        in_pdb     = "✓" if (chain_r, resnum_r) in pdb_set    else "—"
        in_tdls    = "✓" if (chain_r, resnum_r) in tdls_set   else "—"
        in_p2r_pdbe= "✓" if (chain_r, resnum_r) in p2rank_set else "—"
        in_p2r_loc = "✓" if any(r[1]==chain_r and r[2]==resnum_r
                                  for r in p2rank_local_residues) else "—"

        # Get residue name from protein_atoms
        res_name = next(
            (a[0] for a in protein_atoms if a[1]==chain_r and a[2]==resnum_r),
            "?"
        )
        rows.append({
            "Residue"              : res_name,
            "Chain"                : chain_r,
            "Res num"              : resnum_r,
            "PDB contacts"         : in_pdb,
            "3DLigandSite (PDBe-KB)": in_tdls,
            "P2Rank (PDBe-KB)"     : in_p2r_pdbe,
            "P2Rank (local run)"   : in_p2r_loc,
        })

    combined_df = pd.DataFrame(rows)

    if combined_df.empty:
        print("  No residues found from any source.")
    else:
        # Summary counts
        print(f"\n  PDB atomic contacts       : {len(pdb_set)} residues")
        print(f"  3DLigandSite (PDBe-KB)    : {len(tdls_set)} residues")
        print(f"  P2Rank (PDBe-KB)          : {len(p2rank_set)} residues")
        print(f"  P2Rank (local, best pocket): {len(p2rank_local_residues)} residues")

        # Count how many sources agree per residue
        src_cols = ["PDB contacts","3DLigandSite (PDBe-KB)",
                    "P2Rank (PDBe-KB)","P2Rank (local run)"]
        combined_df["Sources agree"] = combined_df[src_cols].apply(
            lambda r: sum(v == "✓" for v in r), axis=1
        )
        combined_df = combined_df.sort_values(
            ["Sources agree","Res num"], ascending=[False, True]
        ).reset_index(drop=True)

        display(combined_df)
        all_output_dfs[lig_label] = combined_df

# ══════════════════════════════════════════════════════════════════════
# BLOCK 5 — Save to Excel (one sheet per ligand)
# ══════════════════════════════════════════════════════════════════════
out_name = f"binding_annotation_{pdb_id}.xlsx"
with pd.ExcelWriter(out_name, engine="openpyxl") as writer:
    lig_summary_df.to_excel(writer, sheet_name="ligands_detected",  index=False)
    if not match_df.empty:
        match_df.to_excel(writer, sheet_name="pocket_ligand_match", index=False)
    for lig_label, df in all_output_dfs.items():
        # Sheet names max 31 chars
        sheet = re.sub(r'[^\w]', '_', lig_label)[:31]
        df.to_excel(writer, sheet_name=sheet, index=False)

print(f"\n\nSaved: {out_name}")
files.download(out_name)

# **3) Cleaning-up for space**

In [ ]:
# @title
import os
import shutil

print("Cleaning up /content except for sample_data...")

# Get a list of all items in /content
items_in_content = os.listdir('/content')

for item in items_in_content:
    # Skip 'sample_data' directory
    if item == 'sample_data':
        continue

    item_path = os.path.join('/content', item)

    try:
        if os.path.isdir(item_path):
            shutil.rmtree(item_path)
            print(f"  Removed directory: {item}")
        elif os.path.isfile(item_path):
            os.remove(item_path)
            print(f"  Removed file: {item}")
    except OSError as e:
        print(f"Error removing {item_path}: {e}")

print("Cleanup complete. Only 'sample_data' should remain.")

# Optionally, verify the contents after cleanup
# !ls -R /content